# NBA Scraping

All the scraping for the project in one notebook. Data comes from basketball-reference.com, seasons 2015-16 to 2025-26 (on the site a season is named by the year it ends, so `2016` = 2015-16).

The order matters: teams come first, because the roster pages are found through the team links.

Every request waits 4 seconds. The site blocks you for a while if you send more than 20 requests per minute.

In [ ]:
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

HEADERS = {'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                         '(KHTML, like Gecko) Chrome/124.0 Safari/537.36'}
BASE = 'https://www.basketball-reference.com'

SEASONS = range(2019, 2027)   #it takes 30 minutes for 5 years


def get_soup(url):
    response = requests.get(url, headers=HEADERS, timeout=10)
    if response.status_code != 200:
        raise Exception(f'{url} -> status {response.status_code}')
    response.encoding = 'utf-8'
    time.sleep(4)
    return BeautifulSoup(response.text, 'html.parser')

## Clubs

One page (`/teams/`) has all 30 active clubs. We only keep the `full_table` rows, the other rows are the old names of the same club (like New Jersey Nets for Brooklyn).

In [2]:
data = get_soup(BASE + '/teams/')

In [3]:
table = data.find('table' , id ='teams_active')
rows = table.find_all( 'tr' , class_='full_table')


col_list =[]
cols= table.find_all('thead')
for col in cols:
    header = col.find_all('th' ,attrs={'data-stat': True})
    for head in header:
        col_name = head.get('data-stat')
        col_list.append(col_name)

res = []
for row in rows:
    team = {'club_name': row.find('a').text}
    team['club_id'] =  row.find('a').get('href').split('/')[2]
    for i in col_list[1:]:
        team[i] = row.find('td', attrs={'data-stat': i}).text

    res.append(team)

clubs = pd.DataFrame(res)
print(len(clubs), 'clubs')

30 clubs


In [4]:
clubs.to_csv('clubs.csv' , index=False)

## Teams per season + champions

From the season summary page we take the two conference standings tables (30 teams) and the league champion from the box at the top of the page.

The club id is not always the same as the season abbreviation (Brooklyn is `NJN` as a club and `BRK` in a season), so `club_id` is added by matching the team name.

In [5]:
def get_teams(soup, year):
    teams = []
    for conf in ['E', 'W']:
        table = soup.find('table', id='confs_standings_' + conf)
        for row in table.find('tbody').find_all('tr'):
            link = row.find('a')
            teams.append({
                'season': year,
                'team_name': link.text,
                'team_abbr': link['href'].split('/')[2],
                'conference': conf,
                'wins': int(row.find('td', {'data-stat': 'wins'}).text),
                'losses': int(row.find('td', {'data-stat': 'losses'}).text),
                'team_url': link['href'],
            })
    return teams


def get_champion(soup):
    for p in soup.find('div', id='meta').find_all('p'):
        if 'League Champion' in p.text:
            return p.find('a')['href'].split('/')[2]

In [6]:
all_teams = []
champions = []

for year in SEASONS:
    soup = get_soup(f'{BASE}/leagues/NBA_{year}.html')

    teams = get_teams(soup, year)
    all_teams.extend(teams)
    champions.append({'season': year, 'champion': get_champion(soup)})

    print(year, '->', len(teams), 'teams')

2019 -> 30 teams
2020 -> 30 teams
2021 -> 30 teams
2022 -> 30 teams
2023 -> 30 teams
2024 -> 30 teams
2025 -> 30 teams
2026 -> 30 teams


In [7]:
team_seasons = pd.DataFrame(all_teams)
team_seasons = team_seasons.merge(clubs[['club_id', 'club_name']], left_on='team_name', right_on='club_name', how='left')
team_seasons = team_seasons.drop(columns='club_name')

print('teams without a club:', team_seasons[team_seasons['club_id'].isna()]['team_name'].unique())

team_seasons.to_csv('team_seasons.csv', index=False)
pd.DataFrame(champions).to_csv('champions.csv', index=False)

teams without a club: <StringArray>
[]
Length: 0, dtype: str


## Rosters

One page per team per season (11 × 30 = 330 pages, around 25 minutes). We go through the `team_url` of every row in `team_seasons`.

Last time this stopped in the middle of 2016 with a `NoneType` error. That happens when the site sends back an error page instead of the team page, so now `get_soup` stops with the status code, and `get_roster` says which page had no roster table.

In [8]:
def get_roster(soup, year, team_abbr):
    players = []
    table = soup.find('table', id='roster')
    if table is None:
        raise Exception(f'no roster table for {team_abbr} {year}')

    for row in table.find('tbody').find_all('tr'):
        player = {'season': year, 'team_abbr': team_abbr}

        for cell in row.find_all(['th', 'td']):
            player[cell['data-stat']] = cell.text.strip()

        link = row.find('a', href=lambda h: h and h.startswith('/players/'))
        player['player_id'] = link['href'].split('/')[-1].replace('.html', '')

        players.append(player)
    return players

In [9]:
all_players = []

for _, team in team_seasons.iterrows():
    soup = get_soup(BASE + team['team_url'])

    roster = get_roster(soup, team['season'], team['team_abbr'])
    all_players.extend(roster)

    print(team['season'], team['team_abbr'], '->', len(roster), 'players')

pd.DataFrame(all_players).to_csv('rosters.csv', index=False)

2019 MIL -> 24 players
2019 TOR -> 22 players
2019 PHI -> 26 players
2019 BOS -> 17 players
2019 IND -> 17 players
2019 BRK -> 19 players
2019 ORL -> 17 players
2019 DET -> 20 players
2019 CHO -> 17 players
2019 MIA -> 18 players
2019 WAS -> 25 players
2019 ATL -> 22 players
2019 CHI -> 22 players
2019 CLE -> 27 players
2019 NYK -> 23 players
2019 GSW -> 17 players
2019 DEN -> 18 players
2019 POR -> 18 players
2019 HOU -> 23 players
2019 UTA -> 18 players
2019 OKC -> 18 players
2019 SAS -> 16 players
2019 LAC -> 22 players
2019 SAC -> 20 players
2019 LAL -> 22 players
2019 MIN -> 21 players
2019 MEM -> 28 players
2019 NOP -> 20 players
2019 DAL -> 21 players
2019 PHO -> 24 players
2020 MIL -> 17 players
2020 TOR -> 18 players
2020 BOS -> 17 players
2020 IND -> 17 players
2020 MIA -> 21 players
2020 PHI -> 19 players
2020 BRK -> 24 players
2020 ORL -> 19 players
2020 CHO -> 16 players
2020 WAS -> 23 players
2020 CHI -> 17 players
2020 NYK -> 17 players
2020 DET -> 22 players
2020 ATL ->

## Player stats: per game and advanced

The last run only had `range(2016, 2017)`, so both files had one season. Now it's every season, and the advanced table id is `advanced` (not `advanced_stats`).

A traded player has one row per team plus a total row (`2TM`, `3TM`). The rows that repeat the header in the middle of the table are skipped.

In [10]:
def get_player_stats(soup, year, table_id):
    players = []
    table = soup.find('table', id=table_id)
    if table is None:
        ids = [t.get('id') for t in soup.find_all('table')]
        raise Exception(f'table {table_id} not found. tables on page: {ids}')

    for row in table.find('tbody').find_all('tr'):
        if 'thead' in row.get('class', []):
            continue

        link = row.find('a', href=lambda h: h and h.startswith('/players/'))
        if link is None:
            continue

        player = {'season': year}
        for cell in row.find_all(['th', 'td']):
            player[cell['data-stat']] = cell.text.strip()
        player['player_id'] = link['href'].split('/')[-1].replace('.html', '')

        players.append(player)
    return players

In [11]:
per_game_rows = []
advanced_rows = []

for year in SEASONS:
    soup = get_soup(f'{BASE}/leagues/NBA_{year}_per_game.html')
    rows = get_player_stats(soup, year, 'per_game_stats')
    per_game_rows.extend(rows)
    print(year, 'per_game ->', len(rows), 'rows')

    soup = get_soup(f'{BASE}/leagues/NBA_{year}_advanced.html')
    rows = get_player_stats(soup, year, 'advanced')
    advanced_rows.extend(rows)
    print(year, 'advanced ->', len(rows), 'rows')

pd.DataFrame(per_game_rows).to_csv('per_game.csv', index=False)
pd.DataFrame(advanced_rows).to_csv('advanced.csv', index=False)

2019 per_game -> 708 rows
2019 advanced -> 708 rows
2020 per_game -> 651 rows
2020 advanced -> 651 rows
2021 per_game -> 705 rows
2021 advanced -> 705 rows
2022 per_game -> 812 rows
2022 advanced -> 812 rows
2023 per_game -> 679 rows
2023 advanced -> 679 rows
2024 per_game -> 735 rows
2024 advanced -> 735 rows
2025 per_game -> 735 rows
2025 advanced -> 735 rows
2026 per_game -> 733 rows
2026 advanced -> 733 rows


## Other tables

Six more tables that all look the same (one row per player, coach or team), so one loop does all of them. The table ids were checked on the 2016 pages before running.

| file | what's in it |
|---|---|
| `mvp_votes` | every player who got MVP votes |
| `totals` | season totals of every player |
| `play_by_play` | % of minutes at each position |
| `playoffs_per_game` | playoff stats |
| `coaches` | coach of every team every season |
| `team_advanced` | advanced team stats, arena and attendance |

The id column is called `id` here, because depending on the table it's a player, a coach or a team.

In [12]:
PAGES = [
    ('mvp_votes',         '/awards/awards_{year}.html',            'mvp',            '/players/'),
    ('totals',            '/leagues/NBA_{year}_totals.html',       'totals_stats',   '/players/'),
    ('play_by_play',      '/leagues/NBA_{year}_play-by-play.html', 'pbp_stats',      '/players/'),
    ('playoffs_per_game', '/playoffs/NBA_{year}_per_game.html',    'per_game_stats', '/players/'),
    ('coaches',           '/leagues/NBA_{year}_coaches.html',      'NBA_coaches',    '/coaches/'),
    ('team_advanced',     '/leagues/NBA_{year}.html',              'advanced-team',  '/teams/'),
]


def get_link_id(href):
    # /players/j/jamesle01.html -> jamesle01
    # /coaches/kerrst01c.html   -> kerrst01c
    # /teams/CLE/2016.html      -> CLE
    parts = href.split('/')
    if parts[1] == 'teams':
        return parts[2]
    return parts[-1].replace('.html', '')


def get_rows(soup, year, table_id, link_prefix):
    table = soup.find('table', id=table_id)
    if table is None:
        ids = [t.get('id') for t in soup.find_all('table')]
        raise Exception(f'table {table_id} not found. tables on page: {ids}')

    rows = []
    for row in table.find('tbody').find_all('tr'):
        if 'thead' in row.get('class', []):
            continue
        link = row.find('a', href=lambda h: h and h.startswith(link_prefix))
        if link is None:
            continue

        data = {'season': year}
        for cell in row.find_all(['th', 'td']):
            data[cell['data-stat']] = cell.text.strip()
        data['id'] = get_link_id(link['href'])
        rows.append(data)
    return rows

In [13]:
for name, page, table_id, link_prefix in PAGES:
    all_rows = []
    for year in SEASONS:
        soup = get_soup(BASE + page.format(year=year))
        rows = get_rows(soup, year, table_id, link_prefix)
        all_rows.extend(rows)
        print(name, year, '->', len(rows), 'rows')
    pd.DataFrame(all_rows).to_csv(name + '.csv', index=False)

mvp_votes 2019 -> 12 rows
mvp_votes 2020 -> 12 rows
mvp_votes 2021 -> 15 rows
mvp_votes 2022 -> 12 rows
mvp_votes 2023 -> 13 rows
mvp_votes 2024 -> 9 rows
mvp_votes 2025 -> 12 rows
mvp_votes 2026 -> 8 rows
totals 2019 -> 708 rows
totals 2020 -> 651 rows
totals 2021 -> 705 rows
totals 2022 -> 812 rows
totals 2023 -> 679 rows
totals 2024 -> 735 rows
totals 2025 -> 735 rows
totals 2026 -> 733 rows
play_by_play 2019 -> 708 rows
play_by_play 2020 -> 651 rows
play_by_play 2021 -> 705 rows
play_by_play 2022 -> 812 rows
play_by_play 2023 -> 679 rows
play_by_play 2024 -> 735 rows
play_by_play 2025 -> 735 rows
play_by_play 2026 -> 733 rows
playoffs_per_game 2019 -> 212 rows
playoffs_per_game 2020 -> 217 rows
playoffs_per_game 2021 -> 239 rows
playoffs_per_game 2022 -> 217 rows
playoffs_per_game 2023 -> 217 rows
playoffs_per_game 2024 -> 214 rows
playoffs_per_game 2025 -> 219 rows
playoffs_per_game 2026 -> 230 rows
coaches 2019 -> 33 rows
coaches 2020 -> 33 rows
coaches 2021 -> 32 rows
coaches 20

## Done

Files written by this notebook:

`clubs.csv`, `team_seasons.csv`, `champions.csv`, `rosters.csv`, `per_game.csv`, `advanced.csv`, `mvp_votes.csv`, `totals.csv`, `play_by_play.csv`, `playoffs_per_game.csv`, `coaches.csv`, `team_advanced.csv`

`teams.csv` is not needed anymore: `team_seasons.csv` has the same columns plus `club_id`. Next step is the cleaning notebook.